In [12]:
import os, time, glob, cv2, numpy as np, pandas as pd
from ultralytics import YOLO

MODEL_PATH = r"C:\Users\luisp\Desktop\VC\prac1\P4\runs\detect\plates_s_1280_rect\weights\best.pt"
TEST_DIR   = r"C:\Users\luisp\Desktop\VC\prac1\P4\matriculas\test\images"
TESSERACT_EXE = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

CONF_THRES = 0.25
IOU_THRES  = 0.45
IMG_SIZE   = 1280
ALLOWLIST  = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
PADDING_PCT = 0.12

import pytesseract
if os.path.isfile(TESSERACT_EXE):
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_EXE

try:
    import torch, easyocr
    use_gpu = bool(getattr(torch, "cuda", None) and torch.cuda.is_available())
    reader = easyocr.Reader(['en'], gpu=use_gpu, verbose=False)
except:
    reader = None

def safe_imread(p):
    a = np.fromfile(p, dtype=np.uint8)
    return cv2.imdecode(a, cv2.IMREAD_COLOR)

def clip(v, lo, hi):
    return max(lo, min(hi, v))

def pad_crop(img, xyxy, pad_pct):
    h, w = img.shape[:2]
    x1, y1, x2, y2 = map(int, xyxy)
    bw, bh = x2 - x1, y2 - y1
    px, py = int(bw * pad_pct), int(bh * pad_pct)
    x1p = clip(x1 - px, 0, w - 1); y1p = clip(y1 - py, 0, h - 1)
    x2p = clip(x2 + px, 0, w - 1); y2p = clip(y2 + py, 0, h - 1)
    return img[y1p:y2p, x1p:x2p]

def preprocess(crop):
    g = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    g = cv2.bilateralFilter(g, 7, 35, 35)
    return cv2.adaptiveThreshold(g,255,cv2.ADAPTIVE_THRESH_MEAN_C,cv2.THRESH_BINARY,31,5)

def best_box(det):
    if det is None or det.boxes is None or len(det.boxes)==0:
        return None
    conf = det.boxes.conf.cpu().numpy()
    return det.boxes.xyxy.cpu().numpy()[int(np.argmax(conf))]

def list_imgs(f):
    exts = ("*.jpg","*.jpeg","*.png","*.bmp","*.tif","*.tiff","*.webp","*.JPG","*.PNG","*.JPEG","*.WEBP")
    r = []
    for e in exts: r += glob.glob(os.path.join(f,e))
    return sorted(r)

def main():
    model = YOLO(MODEL_PATH)
    out_dir = os.path.join(TEST_DIR,"ocr_preview")
    os.makedirs(out_dir, exist_ok=True)
    imgs = list_imgs(TEST_DIR)
    rows = []
    for p in imgs:
        img = safe_imread(p)
        nombre = os.path.basename(p)
        texto_tess, texto_easy = "", ""
        tiempo_tess, tiempo_easy = np.nan, np.nan
        err_tess, err_easy = "", ""
        if img is None:
            rows.append({"nombre_archivo":nombre,"texto_tesseract":texto_tess,"texto_EasyOCR":texto_easy,"tiempo_tesseract":tiempo_tess,"tiempo_EasyOCR":tiempo_easy,"error_tesseract":"imread_failed","error easyOCR":"imread_failed"})
            continue
        try:
            det = model.predict(img, imgsz=IMG_SIZE, conf=CONF_THRES, iou=IOU_THRES, verbose=False)[0]
            bb = best_box(det)
        except Exception as e:
            rows.append({"nombre_archivo":nombre,"texto_tesseract":texto_tess,"texto_EasyOCR":texto_easy,"tiempo_tesseract":tiempo_tess,"tiempo_EasyOCR":tiempo_easy,"error_tesseract":str(e),"error easyOCR":str(e)})
            continue
        if bb is None:
            rows.append({"nombre_archivo":nombre,"texto_tesseract":texto_tess,"texto_EasyOCR":texto_easy,"tiempo_tesseract":tiempo_tess,"tiempo_EasyOCR":tiempo_easy,"error_tesseract":"no_plate","error easyOCR":"no_plate"})
            continue
        crop = pad_crop(img, bb, PADDING_PCT)
        proc = preprocess(crop)
        try:
            t0 = time.perf_counter()
            cfg = f'--oem 3 --psm 7 -c tessedit_char_whitelist={ALLOWLIST}'
            texto_tess = pytesseract.image_to_string(proc, config=cfg).strip().upper().replace(" ","").replace("\n","")
            tiempo_tess = time.perf_counter() - t0
        except Exception as e:
            err_tess = str(e)
        if reader is not None:
            try:
                t1 = time.perf_counter()
                res = reader.readtext(proc, detail=0, paragraph=True, allowlist=ALLOWLIST)
                texto_easy = "".join(res).strip().upper().replace(" ","")
                tiempo_easy = time.perf_counter() - t1
            except Exception as e:
                err_easy = str(e)
        else:
            err_easy = "easyocr_not_available"
        try:
            vis = crop.copy()
            cv2.putText(vis, f"T:{texto_tess}", (10,35), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,0,0),2)
            cv2.putText(vis, f"E:{texto_easy}", (10,70), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0),2)
            cv2.imwrite(os.path.join(out_dir, nombre), vis)
        except:
            pass
        rows.append({"nombre_archivo":nombre,"texto_tesseract":texto_tess,"texto_EasyOCR":texto_easy,"tiempo_tesseract":tiempo_tess,"tiempo_EasyOCR":tiempo_easy,"error_tesseract":err_tess,"error easyOCR":err_easy})
        print(f"{nombre} | Tesseract:{texto_tess} | EasyOCR:{texto_easy} | tT:{tiempo_tess:.4f}s | tE:{tiempo_easy if isinstance(tiempo_easy,float) else np.nan:.4f}s")
    df = pd.DataFrame(rows, columns=["nombre_archivo","texto_tesseract","texto_EasyOCR","tiempo_tesseract","tiempo_EasyOCR","error_tesseract","error easyOCR"])
    out_csv = os.path.join(TEST_DIR,"ocr_comparison.csv")
    df.to_csv(out_csv, index=False, encoding="utf-8")
    print(f"\nCSV guardado en: {out_csv}")

if __name__ == "__main__":
    main()


0116GPD_2959_aug3.jpg | Tesseract:PO | EasyOCR:26175GRD | tT:0.1910s | tE:0.0639s
0116GPD_2959_aug3.jpg | Tesseract:PO | EasyOCR:26175GRD | tT:0.2039s | tE:0.0609s
0116GPD_aug1.jpg | Tesseract:VE | EasyOCR:40116GPD | tT:0.1438s | tE:0.0363s
0116GPD_aug1.jpg | Tesseract:VE | EasyOCR:40116GPD | tT:0.1354s | tE:0.0364s
0116HGV.jpg | Tesseract:SC | EasyOCR:00116XAGV | tT:0.1603s | tE:0.0514s
0116HGV.jpg | Tesseract:SC | EasyOCR:00116XAGV | tT:0.1961s | tE:0.0531s
0290KWT_8386_aug3.jpg | Tesseract:AA230KN1J | EasyOCR:2110290KNT | tT:0.1539s | tE:0.0615s
0290KWT_8386_aug3.jpg | Tesseract:AA230KN1J | EasyOCR:2110290KNT | tT:0.1372s | tE:0.0606s
0290KWT_aug3.jpg | Tesseract: | EasyOCR:0290KWT | tT:0.1399s | tE:0.0534s
0290KWT_aug3.jpg | Tesseract: | EasyOCR:0290KWT | tT:0.2454s | tE:0.0541s
0290KWT_aug5.jpg | Tesseract: | EasyOCR:0290KUT | tT:0.1893s | tE:0.0445s
0290KWT_aug5.jpg | Tesseract: | EasyOCR:0290KUT | tT:0.1935s | tE:0.0454s
0416MLX.jpg | Tesseract: | EasyOCR: | tT:0.3386s | tE:0.25

In [17]:
import os, csv, cv2, re
from collections import defaultdict, Counter, deque
import numpy as np
from ultralytics import YOLO

VIDEO_IN  = r"C:\Users\luisp\Desktop\VC\prac1\P4\videos\C0142.mp4"
VIDEO_OUT = r"C:\Users\luisp\Desktop\VC\prac1\P4\outputs\video_annotado3.mp4"
CSV_OUT   = r"C:\Users\luisp\Desktop\VC\prac1\P4\outputs\detecciones3.csv"
PLATES_SUMMARY_TXT = r"C:\Users\luisp\Desktop\VC\prac1\P4\outputs\matriculas_summary.txt"

detector = YOLO("yolo11n.pt")
plate_model = YOLO(r"C:\Users\luisp\Desktop\VC\prac1\P4\runs\detect\plates_s_1280_rect\weights\best.pt")

TARGET_CLASSES = {"car", "motorbike", "bus", "truck"}
TRACKER = "bytetrack.yaml"
DET_CONF = 0.25

PLATE_CONF   = 0.28
PLATE_IOU    = 0.60
PLATE_IMGSZ  = 1280

PLATE_ONLY_BOTTOM_BAND = True
BOTTOM_FRAC   = 0.65
EXTRA_BAND_UP = 0.10

PLATE_AR_MIN, PLATE_AR_MAX = 3.0, 7.2
ALLOW_TWO_LINE = True
MIN_PLATE_AREA = 400

VEH_PLATE_AREA_FRAC_MIN = 0.0012
VEH_PLATE_AREA_FRAC_MAX = 0.09
PLATE_VH_FRAC_MIN = 0.035
PLATE_VH_FRAC_MAX = 0.30

ANONYMIZE = False
USE_CONTOUR_FALLBACK = True
ANTI_HEADLIGHT = True
FLOW_ANALYSIS = True

HIST_N = 7
ACCEPT_K = 3
HOLD_FRAMES = 10
MISSING_TOLERANCE = 12

ALLOWLIST = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
PLATE_RE = re.compile(r"^[0-9A-Z]{6,8}$")

import torch, easyocr
use_gpu = bool(getattr(torch, "cuda", None) and torch.cuda.is_available())
reader = easyocr.Reader(['en'], gpu=use_gpu, verbose=False)

def clamp_roi(x1, y1, x2, y2, W, H):
    x1 = max(0, min(W, x1)); x2 = max(0, min(W, x2))
    y1 = max(0, min(H, y1)); y2 = max(0, min(H, y2))
    return x1, y1, x2, y2

def upscale(crop, target_min_side=900, max_side=1600):
    h, w = crop.shape[:2]
    s = max(h, w)
    if s < target_min_side:
        r = target_min_side / s
        nw, nh = int(w*r), int(h*r)
        if max(nw, nh) > max_side:
            r = max_side / max(w, h)
            nw, nh = int(w*r), int(h*r)
        return cv2.resize(crop, (nw, nh), interpolation=cv2.INTER_CUBIC)
    return crop

def plausible_plate_absrel(w, h, vw, vh):
    area = w*h
    if area < MIN_PLATE_AREA: return False
    ar = w / max(1, h)
    if ALLOW_TWO_LINE and (1.2 <= ar <= 2.2):
        pass
    elif not (PLATE_AR_MIN <= ar <= PLATE_AR_MAX):
        return False
    rel_area = area / (max(1, vw)*max(1, vh))
    if not (VEH_PLATE_AREA_FRAC_MIN <= rel_area <= VEH_PLATE_AREA_FRAC_MAX):
        return False
    rel_h = h / max(1, vh)
    if not (PLATE_VH_FRAC_MIN <= rel_h <= PLATE_VH_FRAC_MAX):
        return False
    return True

def is_headlight_like(img):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    thr = cv2.threshold(g, 230, 255, cv2.THRESH_BINARY)[1]
    white = thr.mean() / 255.0
    edges = cv2.Canny(g, 80, 160).mean()
    vert = np.abs(cv2.Sobel(g, cv2.CV_64F, 1, 0, ksize=3)).mean()
    return white > 0.65 and edges < 12 and vert < 6

def minarect_warp(bgr):
    if bgr is None or bgr.size == 0: return None
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    g = cv2.GaussianBlur(g, (3,3), 0)
    e = cv2.Canny(g, 50, 150)
    e = cv2.dilate(e, np.ones((3,3), np.uint8), 1)
    cnts,_ = cv2.findContours(e, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return None
    cnt = max(cnts, key=cv2.contourArea)
    rect = cv2.minAreaRect(cnt)
    (cx, cy),(w, h),ang = rect
    if w < 20 or h < 10: return None
    box = cv2.boxPoints(rect).astype(np.float32)
    s = box.sum(axis=1); d = np.diff(box, axis=1).ravel()
    tl = box[np.argmin(s)]; br = box[np.argmax(s)]
    tr = box[np.argmin(d)]; bl = box[np.argmax(d)]
    dw, dh = int(max(w, h)), int(min(w, h))
    if w < h: dw, dh = dh, dw
    dst = np.array([[0,0],[dw-1,0],[dw-1,dh-1],[0,dh-1]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(np.array([tl,tr,br,bl], dtype=np.float32), dst)
    return cv2.warpPerspective(bgr, M, (dw, dh))

def preproc_variants(bgr):
    if bgr is None or bgr.size == 0: return []
    warped = minarect_warp(bgr)
    roi = warped if warped is not None else bgr
    roi = upscale(roi, 900, 1600)
    g = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    out = []
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    g0 = clahe.apply(g)
    _, th0 = cv2.threshold(g0, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    out.append(th0)
    th1 = cv2.adaptiveThreshold(g0,255,cv2.ADAPTIVE_THRESH_MEAN_C,cv2.THRESH_BINARY,31,5)
    out.append(th1)
    bh = cv2.morphologyEx(g, cv2.MORPH_BLACKHAT, np.ones((5,5), np.uint8))
    sx = cv2.Sobel(bh, cv2.CV_8U, 1,0,ksize=3)
    _, th2 = cv2.threshold(sx, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    out.append(th2)
    _, th3 = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    th3 = cv2.morphologyEx(th3, cv2.MORPH_OPEN, np.ones((3,3), np.uint8), iterations=1)
    out.append(th3)
    return out

def rotate3(img):
    h, w = img.shape[:2]
    res = []
    for a in (-5, 0, 5):
        M = cv2.getRotationMatrix2D((w/2, h/2), a, 1.0)
        res.append(cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE))
    return res

def score_text(t):
    s = re.sub(r"\s+", "", t.upper()).replace("O","0") if len(t)<=4 else t.upper()
    s = re.sub(r"[^A-Z0-9]", "", s)
    if not s: return "", -1.0
    base = 0.0
    if PLATE_RE.match(s): base += 0.6
    base += max(0.0, 1.0 - abs(len(s)-7)*0.2)
    return s, base

def ocr_best(img_bin):
    best_txt, best_sc = "", -1.0
    for rot in rotate3(img_bin):
        try:
            z = reader.readtext(rot, detail=0, paragraph=True, allowlist=ALLOWLIST)
            txt = "".join(z).strip().upper().replace(" ","")
        except:
            txt = ""
        norm, sc = score_text(txt)
        if sc > best_sc:
            best_txt, best_sc = norm, sc
    return best_txt

def best_box(det):
    if det is None or det.boxes is None or len(det.boxes) == 0: return None
    confs = det.boxes.conf.cpu().numpy()
    return det.boxes.xyxy.cpu().numpy()[int(np.argmax(confs))]

def position_filters(mx1,my1,mx2,my2, vx1,vy1,vx2,vy2):
    vw = vx2-vx1; vh = vy2-vy1
    vcx = (vx1+vx2)*0.5; pcx = (mx1+mx2)*0.5; pcy = (my1+my2)*0.5
    if abs(pcx - vcx) / max(1, vw*0.5) > 0.6: return False
    if not (vy1 + 0.45*vh <= pcy <= vy1 + 0.95*vh): return False
    return True

def score_plate_abs(px1,py1,px2,py2,pconf, vx1,vy1,vx2,vy2):
    pcx = (px1+px2)*0.5; pcy = (py1+py2)*0.5
    vcx = (vx1+vx2)*0.5; vcy = vy1 + 0.78*(vy2-vy1)
    vw = vx2-vx1; vh = vy2-vy1
    cx_pen = abs(pcx - vcx) / max(1, vw*0.5)
    cy_pen = abs(pcy - vcy) / max(1, vh*0.5)
    return pconf - 0.6*cx_pen - 0.3*cy_pen

plate_history, plate_last_good, plate_hold = {}, {}, {}
text_history, text_final = {}, {}

def push_hist(tid, box_or_none):
    dq = plate_history.setdefault(tid, deque(maxlen=HIST_N))
    dq.append(box_or_none)

def accept_majority(tid):
    dq = plate_history.get(tid, [])
    return sum(1 for b in dq if b is not None) >= ACCEPT_K

def reuse_hold(tid):
    if plate_hold.get(tid,0)>0 and tid in plate_last_good:
        plate_hold[tid] -= 1
        return plate_last_good[tid]
    return None

def push_text(tid, txt):
    if not txt: return
    dq = text_history.setdefault(tid, deque(maxlen=14))
    dq.append(txt)
    c = Counter(dq)
    t,k = c.most_common(1)[0]
    if k >= 3:
        text_final[tid] = t

def detect_plate_roi(frame, rx1,ry1,rx2,ry2, vx1,vy1,vx2,vy2):
    crop = frame[ry1:ry2, rx1:rx2]
    if crop.size == 0: return None
    up = upscale(crop, 900, 1600)
    pr = plate_model.predict(source=up, conf=PLATE_CONF, iou=PLATE_IOU, imgsz=PLATE_IMGSZ, max_det=5, augment=True, agnostic_nms=False, verbose=False)
    sx = crop.shape[1]/up.shape[1]; sy = crop.shape[0]/up.shape[0]
    best, best_sc = None, -1e9
    if pr and len(pr[0].boxes)>0:
        for pb in pr[0].boxes:
            px1,py1,px2,py2 = pb.xyxy[0].tolist()
            pconf = float(pb.conf[0].item())
            px1,py1,px2,py2 = int(px1*sx),int(py1*sy),int(px2*sx),int(py2*sy)
            mx1,my1,mx2,my2 = rx1+px1, ry1+py1, rx1+px2, ry1+py2
            w,h = mx2-mx1, my2-my1
            vw,vh = vx2-vx1, vy2-vy1
            if ANTI_HEADLIGHT:
                pc = frame[my1:my2, mx1:mx2]
                if pc.size and is_headlight_like(pc):
                    continue
            if not plausible_plate_absrel(w,h,vw,vh): 
                continue
            if not position_filters(mx1,my1,mx2,my2, vx1,vy1,vx2,vy2):
                continue
            sc = score_plate_abs(mx1,my1,mx2,my2,pconf, vx1,vy1,vx2,vy2)
            if sc > best_sc:
                best_sc, best = sc, (mx1,my1,mx2,my2,pconf)
    if best is None and USE_CONTOUR_FALLBACK:
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        gray = cv2.bilateralFilter(gray, 9, 75, 75)
        edges = cv2.Canny(gray, 50, 150)
        edges = cv2.dilate(edges, None, iterations=1)
        cnts,_ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        Hc,Wc = crop.shape[:2]
        cand, area_max = None, 0
        for c in cnts:
            x,y,w,h = cv2.boundingRect(c)
            if w<20 or h<10: continue
            if x<2 or y<2 or x+w>Wc-2 or y+h>Hc-2: continue
            area = w*h
            if area>area_max:
                cand, area_max = (x,y,x+w,y+h), area
        if cand:
            px1,py1,px2,py2 = cand
            mx1,my1,mx2,my2 = rx1+px1, ry1+py1, rx1+px2, ry1+py2
            w,h = mx2-mx1, my2-my1
            vw,vh = vx2-vx1, vy2-vy1
            if plausible_plate_absrel(w,h,vw,vh) and position_filters(mx1,my1,mx2,my2, vx1,vy1,vx2,vy2):
                best = (mx1,my1,mx2,my2,0.30)
    return best

def vehicle_bottom_band(x1,y1,x2,y2,H):
    if not PLATE_ONLY_BOTTOM_BAND:
        return x1,y1,x2,y2
    vh = y2-y1
    top = y2 - int(BOTTOM_FRAC*vh) - int(EXTRA_BAND_UP*vh)
    return x1, max(0, top), x2, y2

def main():
    os.makedirs(os.path.dirname(VIDEO_OUT), exist_ok=True)
    os.makedirs(os.path.dirname(CSV_OUT), exist_ok=True)

    cap = cv2.VideoCapture(VIDEO_IN)
    if not cap.isOpened():
        raise FileNotFoundError(VIDEO_IN)

    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter(VIDEO_OUT, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

    csv_f = open(CSV_OUT, "w", newline="", encoding="utf-8")
    cw = csv.writer(csv_f)
    cw.writerow(["fotograma","tipo_objeto","confianza","id_track","x1","y1","x2","y2","flag_plate","conf_plate","mx1","my1","mx2","my2","texto_matricula"])

    seen_ids_by_class = defaultdict(set)
    last_centroid = {}
    exit_side_count = Counter()
    inactive_counter = {}
    already_counted = set()
    unique_vehicle_ids = set()
    collected_plates = []

    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok: break

        gen = detector.track(source=frame, stream=True, persist=True, tracker=TRACKER, conf=DET_CONF, verbose=False)
        try:
            res = next(gen)
        except StopIteration:
            res = None

        if res is None or res.boxes is None or len(res.boxes)==0:
            writer.write(frame)
            frame_idx += 1
            continue

        names = detector.model.names
        boxes = res.boxes
        active_ids = set()

        for b in boxes:
            if b.cls is None or b.conf is None or b.xyxy is None: continue
            cls_id = int(b.cls[0].item())
            conf = float(b.conf[0].item())
            cname = names.get(cls_id, str(cls_id))
            if cname not in TARGET_CLASSES: continue

            x1,y1,x2,y2 = map(int, b.xyxy[0].tolist())
            tid = int(b.id[0].item()) if b.id is not None else -1
            active_ids.add(tid)
            unique_vehicle_ids.add(tid)

            seen_ids_by_class[cname].add(tid)
            cx = int((x1+x2)*0.5); cy = int((y1+y2)*0.5)
            last_centroid[tid] = (cx, cy)

            if not ANONYMIZE:
                cv2.rectangle(frame, (x1,y1),(x2,y2), (0,255,0), 2)
                cv2.putText(frame, f"{cname} {conf:.2f} ID:{tid}", (x1, max(0,y1-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)

            plate_flag, plate_conf = 0, 0.0
            mx1=my1=mx2=my2=0
            plate_txt = ""

            vx1,vy1,vx2,vy2 = x1,y1,x2,y2
            rx1,ry1,rx2,ry2 = vehicle_bottom_band(vx1,vy1,vx2,vy2,H)
            rx1,ry1,rx2,ry2 = clamp_roi(rx1,ry1,rx2,ry2,W,H)

            if rx2>rx1 and ry2>ry1:
                cand = detect_plate_roi(frame, rx1,ry1,rx2,ry2, vx1,vy1,vx2,vy2)
                if cand is not None:
                    bx1,by1,bx2,by2,bconf = cand
                    push_hist(tid, (bx1,by1,bx2,by2,bconf))
                else:
                    push_hist(tid, None)

                g = None
                if accept_majority(tid):
                    dq = plate_history.get(tid, [])
                    last = next((bc for bc in reversed(dq) if bc is not None), None)
                    if last is not None:
                        plate_last_good[tid] = last
                        plate_hold[tid] = HOLD_FRAMES
                        g = last
                else:
                    g = reuse_hold(tid)

                if g:
                    gx1,gy1,gx2,gy2,gc = g
                    plate_flag, plate_conf = 1, gc
                    mx1,my1,mx2,my2 = gx1,gy1,gx2,gy2
                    roi = frame[gy1:gy2, gx1:gx2]
                    best_text = ""
                    for v in preproc_variants(roi):
                        cand_txt = ocr_best(v)
                        if len(cand_txt) >= len(best_text):
                            best_text = cand_txt
                    push_text(tid, best_text)
                    if tid in text_final:
                        plate_txt = text_final[tid]
                        if plate_txt: collected_plates.append(plate_txt)
                    else:
                        plate_txt = best_text
                    if not ANONYMIZE:
                        cv2.rectangle(frame, (gx1, gy1), (gx2, gy2), (255,0,0), 2)
                        label = plate_txt if plate_txt else f"{gc:.2f}"
                        cv2.putText(frame, f"PLATE {label}", (gx1, max(0, gy1-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,0), 2)

            cw.writerow([frame_idx, cname, f"{conf:.3f}", tid, x1,y1,x2,y2, plate_flag, f"{plate_conf:.3f}", mx1,my1,mx2,my2, plate_txt])

        if FLOW_ANALYSIS:
            ids_to_check = set(list(last_centroid.keys()) + list(inactive_counter.keys()))
            for gid in list(ids_to_check):
                if gid in active_ids:
                    inactive_counter[gid] = 0
                else:
                    inactive_counter[gid] = inactive_counter.get(gid, 0) + 1
                    if inactive_counter[gid] == MISSING_TOLERANCE and gid not in already_counted:
                        cx, cy = last_centroid.get(gid, (None, None))
                        if cx is not None:
                            d = {"left": cx, "right": W-cx, "top": cy, "bottom": H-cy}
                            side = min(d, key=d.get)
                            exit_side_count[side] += 1
                            already_counted.add(gid)
                        last_centroid.pop(gid, None)

        cv2.putText(frame, f"Vehiculos detectados: {len({i for i in unique_vehicle_ids if i!=-1})}", (12, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (50,220,50), 2)
        if collected_plates:
            tail = ",".join(collected_plates[-3:])
            cv2.putText(frame, f"Ultimas placas: {tail}", (12, 56), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (50,180,255), 2)

        writer.write(frame)
        frame_idx += 1

    cap.release()
    writer.release()
    csv_f.close()

    unique_ids = len({i for i in unique_vehicle_ids if i!=-1})
    unique_plates = []
    seen = set()
    for p in collected_plates:
        if p and p not in seen:
            unique_plates.append(p); seen.add(p)

    with open(PLATES_SUMMARY_TXT, "w", encoding="utf-8") as f:
        f.write(f"Vehiculos unicos: {unique_ids}\n")
        f.write(f"Placas leidas ({len(unique_plates)}):\n")
        for p in unique_plates:
            f.write(p+"\n")

    print(f"Video: {VIDEO_OUT}")
    print(f"CSV:   {CSV_OUT}")
    print(f"Resumen: {PLATES_SUMMARY_TXT}")
    print(f"Vehiculos unicos: {unique_ids}")
    print(f"Placas unicas: {len(unique_plates)}")

if __name__ == "__main__":
    main()


Video: C:\Users\luisp\Desktop\VC\prac1\P4\outputs\video_annotado3.mp4
CSV:   C:\Users\luisp\Desktop\VC\prac1\P4\outputs\detecciones3.csv
Resumen: C:\Users\luisp\Desktop\VC\prac1\P4\outputs\matriculas_summary.txt
Vehiculos unicos: 280
Placas unicas: 17
